# 06 — Window Functions

**What you will learn:**
- What a Window (frame) is and how to define one
- Ranking functions: `row_number`, `rank`, `dense_rank`, `percent_rank`, `ntile`
- Lead & Lag: accessing previous/next row values
- Running totals & rolling aggregations: `sum`, `avg`, `min`, `max` over a window
- Frame specification: `rowsBetween`, `rangeBetween`
- Practical use cases: running total, top-N per group, period-over-period comparison

## Setup

In [ ]:
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank, percent_rank, ntile,
    lag, lead,
    sum, avg, min, max, count,
    round
)
from pyspark.sql.window import Window

data = [
    ("Engineering", "Alice",   95000.0,  "2023-01"),
    ("Engineering", "Charlie", 105000.0, "2023-01"),
    ("Engineering", "Frank",   88000.0,  "2023-01"),
    ("Engineering", "Hank",    92000.0,  "2023-01"),
    ("Marketing",   "Bob",     72000.0,  "2023-01"),
    ("Marketing",   "Eve",     78000.0,  "2023-01"),
    ("Marketing",   "Ivy",     75000.0,  "2023-01"),
    ("HR",          "Diana",   68000.0,  "2023-01"),
    ("HR",          "Grace",   71000.0,  "2023-01"),
]

df = spark.createDataFrame(data, ["department", "name", "salary", "month"])
df.show()

## 1. Defining a Window Specification

A Window specifies:
- **`partitionBy`** — which group to compute within (like GROUP BY, but keeps all rows)
- **`orderBy`** — sort order within the partition
- **Frame** (optional) — which rows relative to current row to include

In [ ]:
# Window partitioned by department, ordered by salary descending
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

## 2. row_number() — Unique Sequential Number

Assigns 1, 2, 3, … to each row within the partition.
Ties broken by row order (non-deterministic if order columns have equal values).

In [ ]:
df_rn = df.withColumn("row_num", row_number().over(window_dept))
df_rn.orderBy("department", "row_num").show()

## 3. rank() — Rank with Gaps

Tied rows get the same rank; the next rank skips numbers.
e.g., 1, 2, 2, 4 (rank 3 skipped)

In [ ]:
df_rank = df.withColumn("rank", rank().over(window_dept))
df_rank.orderBy("department", "rank").show()

## 4. dense_rank() — Rank Without Gaps

Tied rows get the same rank; next rank is consecutive.
e.g., 1, 2, 2, 3

In [ ]:
# Add a tie to show the difference
tie_data = data + [("Engineering", "NewEng", 105000.0, "2023-01")]  # tie with Charlie
df_tie = spark.createDataFrame(tie_data, ["department", "name", "salary", "month"])

window_tie = Window.partitionBy("department").orderBy(col("salary").desc())

df_ranks = df_tie.withColumn("rank",       rank().over(window_tie)) \
                 .withColumn("dense_rank", dense_rank().over(window_tie)) \
                 .withColumn("row_number", row_number().over(window_tie))

df_ranks.filter(col("department") == "Engineering").orderBy("rank").show()

## 5. Top-N per Group Using row_number()

In [ ]:
# Top 2 highest-paid employees per department
df_top2 = (
    df.withColumn("rn", row_number().over(window_dept))
      .filter(col("rn") <= 2)
      .drop("rn")
)
df_top2.orderBy("department").show()

## 6. percent_rank() — Relative Rank as 0.0 to 1.0

In [ ]:
df_pct = df.withColumn("pct_rank", round(percent_rank().over(window_dept), 3))
df_pct.orderBy("department", col("salary").desc()).show()

## 7. ntile(n) — Divide Partition into N Equal Buckets

In [ ]:
# Divide each department into salary quartiles (4 buckets)
df_ntile = df.withColumn("quartile", ntile(4).over(window_dept))
df_ntile.orderBy("department", col("salary").desc()).show()

## 8. lag() and lead() — Access Adjacent Row Values

| Function | Returns |
|---|---|
| `lag(col, n, default)` | Value of `col` from n rows BEFORE current row |
| `lead(col, n, default)` | Value of `col` from n rows AFTER current row |

In [ ]:
# Monthly sales data to show period-over-period comparison
monthly = [
    ("Engineering", "2023-01", 280000.0),
    ("Engineering", "2023-02", 295000.0),
    ("Engineering", "2023-03", 310000.0),
    ("Engineering", "2023-04", 305000.0),
    ("Marketing",   "2023-01", 220000.0),
    ("Marketing",   "2023-02", 235000.0),
    ("Marketing",   "2023-03", 228000.0),
    ("Marketing",   "2023-04", 245000.0),
]
df_monthly = spark.createDataFrame(monthly, ["department", "month", "total_salary"])

window_time = Window.partitionBy("department").orderBy("month")

df_laglead = df_monthly.withColumn(
    "prev_month_salary", lag("total_salary", 1).over(window_time)
).withColumn(
    "next_month_salary", lead("total_salary", 1).over(window_time)
).withColumn(
    "mom_change", round(col("total_salary") - col("prev_month_salary"), 2)
)

df_laglead.show()

## 9. Running Total (Cumulative Sum)

In [ ]:
# Running total of salary ordered by month, within each department
window_running = (
    Window.partitionBy("department")
          .orderBy("month")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_running = df_monthly.withColumn(
    "running_total", sum("total_salary").over(window_running)
)
df_running.show()

## 10. Rolling Average (Moving Average)

In [ ]:
# 3-month rolling average (current row + 2 preceding)
window_rolling = (
    Window.partitionBy("department")
          .orderBy("month")
          .rowsBetween(-2, Window.currentRow)   # 2 preceding rows + current
)

df_rolling = df_monthly.withColumn(
    "rolling_avg_3m", round(avg("total_salary").over(window_rolling), 2)
)
df_rolling.show()

## 11. Cumulative Min / Max

In [ ]:
df_cumulative = df_monthly.withColumn(
    "cumulative_max", max("total_salary").over(window_running)
).withColumn(
    "cumulative_min", min("total_salary").over(window_running)
)
df_cumulative.show()

## 12. Window Without partitionBy — Entire DataFrame as One Partition

In [ ]:
# Rank ALL employees by salary (no partition = one global partition)
window_global = Window.orderBy(col("salary").desc())

df_global_rank = df.withColumn("global_rank", row_number().over(window_global))
df_global_rank.show()

## 13. Frame Specification Reference

```python
# ROWS BETWEEN — physical row offset
Window.rowsBetween(Window.unboundedPreceding, Window.currentRow)  # cumulative
Window.rowsBetween(-2, 0)                                          # 3-row rolling
Window.rowsBetween(Window.currentRow, Window.unboundedFollowing)   # suffix agg
Window.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)  # full partition

# RANGE BETWEEN — logical value offset (useful for date ranges)
Window.rangeBetween(-6, 0)  # within 6 units of current value
```

## Key Takeaways

| Function | Use Case |
|---|---|
| `row_number()` | Unique sequence; use for Top-N per group |
| `rank()` | Rank with gaps for ties |
| `dense_rank()` | Rank without gaps for ties |
| `percent_rank()` | Relative percentile 0.0–1.0 |
| `ntile(n)` | Split into N equal-sized buckets |
| `lag(col, n)` | Previous row value (period-over-period) |
| `lead(col, n)` | Next row value (look-ahead) |
| `sum().over(running_window)` | Cumulative running total |
| `avg().over(rolling_window)` | Moving/rolling average |

**Window spec pattern:**
```python
w = Window.partitionBy("group_col").orderBy("sort_col")
df.withColumn("result", func().over(w))
```